# 103 — TCGA RNA-seq Download Validation

## Objective

Validate the completeness and local integrity of the downloaded TCGA primary-tumor RNA-seq STAR-count files against the version-controlled frozen GDC manifest.

This notebook:

- inventories the local GDC download directory;
- compares downloaded UUID directories with the expected file identifiers;
- verifies the presence, filename, and size of every manifest-listed payload;
- identifies missing, unexpected, incomplete, or structurally inconsistent files;
- establishes whether the RNA-seq acquisition is complete before downstream processing.

## Scope

This notebook performs acquisition-level validation only. It does not parse gene-count tables, transform expression values, construct an expression matrix, or define the final analytical tumor cohort.

Raw GDC files remain immutable.

In [1]:
# =============================================================================
# 103 — TCGA RNA-seq Download Validation
# Imports, project-root detection, and input paths
# =============================================================================

import hashlib
import sys
from pathlib import Path

import pandas as pd

# Allow imports from src/ when running this notebook from a notebooks subdirectory.
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import Paths

# Version-controlled expected file cohort.
FROZEN_MANIFEST_PATH = (
    Paths.config
    / "manifests"
    / "tcga_rna"
    / "gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt"
)

# Local GDC download directory containing one UUID directory per manifest file.
DOWNLOAD_DIR = Paths.tcga / "star_counts"

print(f"Project root:    {PROJECT_ROOT}")
print(f"Frozen manifest: {FROZEN_MANIFEST_PATH}")
print(f"Download root:   {DOWNLOAD_DIR}")

Project root:    C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics
Frozen manifest: C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt
Download root:   C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\star_counts


In [3]:
# =============================================================================
# Load and validate download-validation inputs
# =============================================================================

if not FROZEN_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Frozen GDC manifest not found: {FROZEN_MANIFEST_PATH}"
    )

if not DOWNLOAD_DIR.is_dir():
    raise NotADirectoryError(
        f"TCGA STAR Counts download directory not found: {DOWNLOAD_DIR}"
    )

manifest_df = pd.read_csv(
    FROZEN_MANIFEST_PATH,
    sep="\t",
    dtype="string",
)

required_manifest_columns = {"id", "filename", "md5", "size"}
missing_manifest_columns = required_manifest_columns - set(manifest_df.columns)

if missing_manifest_columns:
    raise ValueError(
        "Frozen manifest is missing required columns: "
        + ", ".join(sorted(missing_manifest_columns))
    )

if manifest_df.empty:
    raise ValueError("Frozen manifest contains no file records.")

null_counts = manifest_df[list(required_manifest_columns)].isna().sum()
columns_with_nulls = null_counts[null_counts > 0]

if not columns_with_nulls.empty:
    raise ValueError(
        "Frozen manifest contains missing required values: "
        + ", ".join(
            f"{column}={count:,}"
            for column, count in columns_with_nulls.items()
        )
    )

manifest_df["size"] = pd.to_numeric(
    manifest_df["size"],
    errors="raise",
).astype("int64")

duplicate_file_ids = manifest_df["id"].duplicated(keep=False)

if duplicate_file_ids.any():
    raise ValueError(
        "Frozen manifest contains duplicated file IDs: "
        f"{duplicate_file_ids.sum():,} affected rows."
    )

invalid_sizes = manifest_df["size"] <= 0

if invalid_sizes.any():
    raise ValueError(
        "Frozen manifest contains non-positive file sizes: "
        f"{invalid_sizes.sum():,} affected rows."
    )

invalid_md5 = ~manifest_df["md5"].str.fullmatch(
    r"[0-9a-fA-F]{32}",
    na=False,
)

if invalid_md5.any():
    raise ValueError(
        "Frozen manifest contains invalid MD5 values: "
        f"{invalid_md5.sum():,} affected rows."
    )

expected_total_size_gib = manifest_df["size"].sum() / 1024**3

print("Download-validation inputs loaded.")
print(f"Manifest rows:          {len(manifest_df):,}")
print(f"Unique file IDs:        {manifest_df['id'].nunique():,}")
print(f"Expected total size:    {expected_total_size_gib:,.3f} GiB")
print(f"Download directory:     {DOWNLOAD_DIR}")

Download-validation inputs loaded.
Manifest rows:          10,308
Unique file IDs:        10,308
Expected total size:    40.673 GiB
Download directory:     C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\star_counts


In [6]:
# =============================================================================
# Inventory local GDC download structure
# =============================================================================

top_level_entries = list(DOWNLOAD_DIR.iterdir())

download_folder_paths = sorted(
    path
    for path in top_level_entries
    if path.is_dir()
)

root_level_file_paths = sorted(
    path
    for path in top_level_entries
    if path.is_file()
)

other_root_entries = [
    path
    for path in top_level_entries
    if not path.is_dir() and not path.is_file()
]

folder_records = []
local_file_records = []
nested_directory_records = []

for folder_path in download_folder_paths:
    child_entries = list(folder_path.iterdir())

    child_files = [
        path
        for path in child_entries
        if path.is_file()
    ]

    child_directories = [
        path
        for path in child_entries
        if path.is_dir()
    ]

    folder_records.append(
        {
            "folder_id": folder_path.name,
            "folder_path": folder_path,
            "n_files": len(child_files),
            "n_subdirectories": len(child_directories),
        }
    )

    for file_path in child_files:
        local_file_records.append(
            {
                "folder_id": folder_path.name,
                "file_name": file_path.name,
                "local_path": file_path,
                "local_size_bytes": file_path.stat().st_size,
            }
        )

    for directory_path in child_directories:
        nested_directory_records.append(
            {
                "parent_folder_id": folder_path.name,
                "directory_name": directory_path.name,
                "directory_path": directory_path,
            }
        )

folder_inventory_df = pd.DataFrame(
    folder_records,
    columns=[
        "folder_id",
        "folder_path",
        "n_files",
        "n_subdirectories",
    ],
)

local_file_inventory_df = pd.DataFrame(
    local_file_records,
    columns=[
        "folder_id",
        "file_name",
        "local_path",
        "local_size_bytes",
    ],
)

nested_directory_inventory_df = pd.DataFrame(
    nested_directory_records,
    columns=[
        "parent_folder_id",
        "directory_name",
        "directory_path",
    ],
)

observed_total_size_gib = (
    local_file_inventory_df["local_size_bytes"].sum() / 1024**3
)

files_per_folder_summary = (
    folder_inventory_df["n_files"]
    .value_counts()
    .sort_index()
    .rename_axis("files_per_folder")
    .reset_index(name="folder_count")
)

print("Local download structure inventoried.")
print(f"Top-level directories:      {len(download_folder_paths):,}")
print(f"Top-level files:            {len(root_level_file_paths):,}")
print(f"Files inside directories:   {len(local_file_inventory_df):,}")
print(f"Nested directories:         {len(nested_directory_inventory_df):,}")
print(f"Other top-level entries:    {len(other_root_entries):,}")
print(f"Observed direct-file size:  {observed_total_size_gib:,.3f} GiB")
print(f"Manifest expected size:     {expected_total_size_gib:,.3f} GiB")

print("\nDirect files per top-level directory:")
display(files_per_folder_summary)

if root_level_file_paths:
    print("\nTop-level files:")
    print(
        pd.DataFrame(
            {
                "file_name": [path.name for path in root_level_file_paths],
                "file_size_bytes": [
                    path.stat().st_size
                    for path in root_level_file_paths
                ],
            }
        )
    )

if not nested_directory_inventory_df.empty:
    print("\nFirst nested directories:")
    display(nested_directory_inventory_df.head(10))

Local download structure inventoried.
Top-level directories:      10,308
Top-level files:            0
Files inside directories:   11,911
Nested directories:         10,308
Other top-level entries:    0
Observed direct-file size:  40.673 GiB
Manifest expected size:     40.673 GiB

Direct files per top-level directory:


,files_per_folder,folder_count
0,1,8705
1,2,1603



First nested directories:


,parent_folder_id,directory_name,directory_path
0,0007888f-8d96-4c01-8251-7fef6cc71596,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
1,000b6b94-572d-4d06-a8f4-2e43829f83d4,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
2,000d81dd-9ba4-4852-9090-2bf22f6483f0,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
3,0016a232-4573-4e00-afdc-7b4144101ef9,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
4,00174492-1adc-4310-8147-a622befbb466,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
5,0019c951-16c5-48d0-85c8-58d96b12d330,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
6,0022cd20-f64f-4773-b9ff-a3de0b71b259,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
7,0024696f-27f5-4d5d-a933-d8c8c52146a6,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
8,002cf74b-098b-4de6-9a76-0d889c894ab6,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...
9,00362664-cb13-4998-a26c-618dba69ea00,logs,C:\Users\paula\OneDrive\Documentos\Proyectos\p...


In [7]:
# =============================================================================
# Compare expected and observed download structure
# =============================================================================

expected_files_df = manifest_df[
    ["id", "filename", "md5", "size"]
].rename(
    columns={
        "id": "file_id",
        "filename": "expected_file_name",
        "md5": "expected_md5",
        "size": "expected_size_bytes",
    }
)

expected_folder_ids = set(expected_files_df["file_id"])
observed_folder_ids = set(folder_inventory_df["folder_id"])

missing_folder_ids = sorted(
    expected_folder_ids - observed_folder_ids
)

unexpected_folder_ids = sorted(
    observed_folder_ids - expected_folder_ids
)

payload_validation_df = expected_files_df.merge(
    local_file_inventory_df,
    left_on=["file_id", "expected_file_name"],
    right_on=["folder_id", "file_name"],
    how="left",
    validate="one_to_one",
)

payload_validation_df["payload_present"] = (
    payload_validation_df["local_path"].notna()
)

payload_validation_df["size_matches"] = (
    payload_validation_df["payload_present"]
    & (
        payload_validation_df["expected_size_bytes"]
        == payload_validation_df["local_size_bytes"]
    )
)

missing_payloads_df = payload_validation_df.loc[
    ~payload_validation_df["payload_present"]
].copy()

size_mismatches_df = payload_validation_df.loc[
    payload_validation_df["payload_present"]
    & ~payload_validation_df["size_matches"]
].copy()

expected_local_keys_df = expected_files_df[
    ["file_id", "expected_file_name"]
].rename(
    columns={
        "file_id": "folder_id",
        "expected_file_name": "file_name",
    }
)

extra_local_files_df = (
    local_file_inventory_df.merge(
        expected_local_keys_df,
        on=["folder_id", "file_name"],
        how="left",
        indicator=True,
        validate="one_to_one",
    )
    .loc[lambda df: df["_merge"] == "left_only"]
    .drop(columns="_merge")
    .copy()
)

partial_like_files_df = local_file_inventory_df.loc[
    local_file_inventory_df["file_name"].str.endswith(
        (".partial", ".part", ".tmp"),
        na=False,
    )
].copy()

expected_total_size_bytes = int(
    expected_files_df["expected_size_bytes"].sum()
)

matched_payload_size_bytes = int(
    payload_validation_df.loc[
        payload_validation_df["payload_present"],
        "local_size_bytes",
    ].sum()
)

extra_total_size_bytes = int(
    extra_local_files_df["local_size_bytes"].sum()
)

print("Expected-versus-observed comparison completed.")
print(f"Expected folders:          {len(expected_folder_ids):,}")
print(f"Observed folders:          {len(observed_folder_ids):,}")
print(f"Missing folders:           {len(missing_folder_ids):,}")
print(f"Unexpected folders:        {len(unexpected_folder_ids):,}")
print(f"Expected payloads present: {payload_validation_df['payload_present'].sum():,}")
print(f"Missing payloads:          {len(missing_payloads_df):,}")
print(f"Size mismatches:           {len(size_mismatches_df):,}")
print(f"Extra direct files:        {len(extra_local_files_df):,}")
print(f"Partial-like files:        {len(partial_like_files_df):,}")
print(f"Expected payload size:     {expected_total_size_bytes / 1024**3:,.6f} GiB")
print(f"Observed payload size:     {matched_payload_size_bytes / 1024**3:,.6f} GiB")
print(f"Extra direct-file size:    {extra_total_size_bytes / 1024**2:,.3f} MiB")

if not extra_local_files_df.empty:
    extra_file_type_summary = (
        extra_local_files_df.assign(
            extension=extra_local_files_df["file_name"].map(
                lambda name: Path(name).suffix.lower() or "<no extension>"
            )
        )
        .groupby("extension", as_index=False)
        .agg(
            file_count=("file_name", "size"),
            total_size_bytes=("local_size_bytes", "sum"),
        )
    )

    extra_file_type_summary["total_size_mib"] = (
        extra_file_type_summary["total_size_bytes"] / 1024**2
    )

    print("\nExtra direct files by extension:")
    display(extra_file_type_summary)

    print("\nFirst extra direct files:")
    display(
        extra_local_files_df[
            [
                "folder_id",
                "file_name",
                "local_size_bytes",
            ]
        ].head(20)
    )

Expected-versus-observed comparison completed.
Expected folders:          10,308
Observed folders:          10,308
Missing folders:           0
Unexpected folders:        0
Expected payloads present: 10,308
Missing payloads:          0
Size mismatches:           0
Extra direct files:        1,603
Partial-like files:        0
Expected payload size:     40.672680 GiB
Observed payload size:     40.672680 GiB
Extra direct-file size:    0.827 MiB

Extra direct files by extension:


,extension,file_count,total_size_bytes,total_size_mib
0,.txt,1603,867148,0.826977



First extra direct files:


,folder_id,file_name,local_size_bytes
20,0058f6ab-2114-4ead-af5e-ba002b5f9cc2,annotations.txt,765
22,006d3450-dab8-487d-a2f2-50fdd64c0a6b,annotations.txt,765
28,008b9e19-c1d3-4c9c-af36-791ec4cc27e8,annotations.txt,609
30,008d6fe7-2139-4fe2-b508-eb4217d77ba5,annotations.txt,609
39,00bc0fe4-481a-4893-b57e-d5113a6bcce2,annotations.txt,560
48,00efbb37-cb7e-4411-b857-eee5a36a3cce,annotations.txt,320
52,00fdfb77-4072-4b79-8b7c-8ed8c8fdc01e,annotations.txt,609
54,0101462d-6b47-4134-8164-667ebddc9091,annotations.txt,510
57,01095104-f0a7-402f-b10f-c7bbee6171f5,annotations.txt,574
60,012adb3a-6b22-4476-96fd-b5470d062e4e,annotations.txt,609


In [9]:
# =============================================================================
# Characterize auxiliary GDC annotation files
# =============================================================================

extra_file_name_summary = (
    extra_local_files_df
    .groupby("file_name", as_index=False)
    .agg(
        file_count=("file_name", "size"),
        total_size_bytes=("local_size_bytes", "sum"),
    )
)

extra_file_name_summary["total_size_mib"] = (
    extra_file_name_summary["total_size_bytes"] / 1024**2
)

print("Auxiliary direct files by exact filename:")
display(extra_file_name_summary)

unexpected_auxiliary_files_df = extra_local_files_df.loc[
    extra_local_files_df["file_name"] != "annotations.txt"
].copy()

if not unexpected_auxiliary_files_df.empty:
    raise ValueError(
        "Direct auxiliary files other than annotations.txt were detected."
    )

annotation_files_df = extra_local_files_df.loc[
    extra_local_files_df["file_name"] == "annotations.txt"
].copy()

annotation_structure_records = []

for row in annotation_files_df.itertuples(index=False):
    annotation_path = Path(row.local_path)

    annotation_text = annotation_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    annotation_lines = annotation_text.splitlines()

    annotation_structure_records.append(
        {
            "folder_id": row.folder_id,
            "annotation_path": annotation_path,
            "n_lines": len(annotation_lines),
            "first_line": (
                annotation_lines[0]
                if annotation_lines
                else "<empty>"
            ),
            "content": annotation_text,
        }
    )

annotation_structure_df = pd.DataFrame(annotation_structure_records)

line_count_summary = (
    annotation_structure_df["n_lines"]
    .value_counts()
    .sort_index()
    .rename_axis("lines_per_file")
    .reset_index(name="file_count")
)

first_line_summary = (
    annotation_structure_df["first_line"]
    .value_counts()
    .rename_axis("first_line")
    .reset_index(name="file_count")
)

print(f"Annotation files:       {len(annotation_structure_df):,}")
print(
    "Empty annotation files:",
    f"{(annotation_structure_df['n_lines'] == 0).sum():,}",
)

print("\nLines per annotation file:")
display(line_count_summary)

print("\nMost frequent first lines:")
display(first_line_summary.head(10))

example_positions = sorted(
    {
        0,
        len(annotation_structure_df) // 2,
        len(annotation_structure_df) - 1,
    }
)

print("\nAnnotation content examples:")

for position in example_positions:
    example = annotation_structure_df.iloc[position]

    print("\n" + "=" * 80)
    print(f"Folder ID: {example['folder_id']}")
    print("=" * 80)
    print(example["content"])

Auxiliary direct files by exact filename:


,file_name,file_count,total_size_bytes,total_size_mib
0,annotations.txt,1603,867148,0.826977


Annotation files:       1,603
Empty annotation files: 0

Lines per annotation file:


,lines_per_file,file_count
0,2,700
1,3,802
2,4,66
3,5,31
4,6,4



Most frequent first lines:


,first_line,file_count
0,id\tsubmitter_id\tentity_type\tentity_id\tcate...,1603



Annotation content examples:

Folder ID: 0058f6ab-2114-4ead-af5e-ba002b5f9cc2
id	submitter_id	entity_type	entity_id	category	classification	created_datetime	status	notes
e4d75445-d7cd-547a-a0db-e5b964d42ae6	8806	aliquot	845d11ec-7fc7-4498-a0f4-1187413c5914	Center QC failed	CenterNotification	2012-07-20T00:00:00	Approved	RNA-seq:LOW 5/3 COVERAGE RATIO
3258d1b6-be3c-54ca-b3d8-22612b85efb1	29754	aliquot	845d11ec-7fc7-4498-a0f4-1187413c5914	Item flagged DNU	CenterNotification	2015-12-11T00:00:00	Approved	QC Warning- Expression quality metrics (e.g. 3-prime bias, high duplicate rate, genomic contamination, or low alignment rate) are well below average for this file, which should be used with caution, if at all, for expression analysis. However, other analyses, e.g. mutation and structural variant calling, may still give reliable result.

Folder ID: 82cdc03b-fbae-42da-a7e8-416839910b94
id	submitter_id	entity_type	entity_id	category	classification	created_datetime	status	notes
19b43312-23d8-

In [11]:
# =============================================================================
# Parse and validate GDC annotations
# =============================================================================

expected_annotation_columns = [
    "id",
    "submitter_id",
    "entity_type",
    "entity_id",
    "category",
    "classification",
    "created_datetime",
    "status",
    "notes",
]

annotation_tables = []

for row in annotation_files_df.itertuples(index=False):
    annotation_path = Path(row.local_path)

    annotation_df = pd.read_csv(
        annotation_path,
        sep="\t",
        dtype="string",
        keep_default_na=False,
    )

    missing_columns = (
        set(expected_annotation_columns)
        - set(annotation_df.columns)
    )

    unexpected_columns = (
        set(annotation_df.columns)
        - set(expected_annotation_columns)
    )

    if missing_columns or unexpected_columns:
        raise ValueError(
            f"Unexpected annotation structure in {annotation_path}. "
            f"Missing columns: {sorted(missing_columns)}. "
            f"Unexpected columns: {sorted(unexpected_columns)}."
        )

    annotation_df = annotation_df[expected_annotation_columns].copy()

    annotation_df.insert(
        0,
        "file_id",
        row.folder_id,
    )

    annotation_df["source_path"] = (
        annotation_path.resolve()
        .relative_to(Paths.root.resolve())
        .as_posix()
    )

    annotation_tables.append(annotation_df)

file_annotations_df = pd.concat(
    annotation_tables,
    ignore_index=True,
).rename(
    columns={
        "id": "annotation_id",
    }
)

file_annotations_df = file_annotations_df[
    [
        "file_id",
        "annotation_id",
        "submitter_id",
        "entity_type",
        "entity_id",
        "category",
        "classification",
        "created_datetime",
        "status",
        "notes",
        "source_path",
    ]
]

expected_annotation_rows = int(
    (annotation_structure_df["n_lines"] - 1).sum()
)

if len(file_annotations_df) != expected_annotation_rows:
    raise ValueError(
        "Parsed annotation-row count does not match the "
        "annotation file structure inventory."
    )

duplicated_file_annotation_links = (
    file_annotations_df.duplicated(
        subset=["file_id", "annotation_id"],
        keep=False,
    )
)

if duplicated_file_annotation_links.any():
    raise ValueError(
        "Duplicated file-to-annotation links detected: "
        f"{duplicated_file_annotation_links.sum():,} affected rows."
    )

print("GDC annotations parsed and validated.")
print(
    f"Annotation-file associations: "
    f"{len(file_annotations_df):,}"
)
print(
    f"Unique annotation IDs:        "
    f"{file_annotations_df['annotation_id'].nunique():,}"
)
print(
    f"Annotated RNA-seq files:       "
    f"{file_annotations_df['file_id'].nunique():,}"
)
print(
    f"Annotated entities:            "
    f"{file_annotations_df['entity_id'].nunique():,}"
)

display(file_annotations_df.head())

GDC annotations parsed and validated.
Annotation-file associations: 2,646
Unique annotation IDs:        2,614
Annotated RNA-seq files:       1,603
Annotated entities:            1,676


,file_id,annotation_id,submitter_id,entity_type,entity_id,category,classification,created_datetime,status,notes,source_path
0,0058f6ab-2114-4ead-af5e-ba002b5f9cc2,e4d75445-d7cd-547a-a0db-e5b964d42ae6,8806,aliquot,845d11ec-7fc7-4498-a0f4-1187413c5914,Center QC failed,CenterNotification,2012-07-20T00:00:00,Approved,RNA-seq:LOW 5/3 COVERAGE RATIO,data/raw/tcga/star_counts/0058f6ab-2114-4ead-a...
1,0058f6ab-2114-4ead-af5e-ba002b5f9cc2,3258d1b6-be3c-54ca-b3d8-22612b85efb1,29754,aliquot,845d11ec-7fc7-4498-a0f4-1187413c5914,Item flagged DNU,CenterNotification,2015-12-11T00:00:00,Approved,QC Warning- Expression quality metrics (e.g. 3...,data/raw/tcga/star_counts/0058f6ab-2114-4ead-a...
2,006d3450-dab8-487d-a2f2-50fdd64c0a6b,36ca9240-0615-524f-bb68-96c3e3c636d8,8761,aliquot,1a59f822-d8fc-4c7a-ad7b-b1b84078e09d,Center QC failed,CenterNotification,2012-07-20T00:00:00,Approved,RNA-seq:LOW 5/3 COVERAGE RATIO,data/raw/tcga/star_counts/006d3450-dab8-487d-a...
3,006d3450-dab8-487d-a2f2-50fdd64c0a6b,50402917-7958-5cb4-824f-855c2af5b98f,29692,aliquot,1a59f822-d8fc-4c7a-ad7b-b1b84078e09d,Item flagged DNU,CenterNotification,2015-12-11T00:00:00,Approved,QC Warning- Expression quality metrics (e.g. 3...,data/raw/tcga/star_counts/006d3450-dab8-487d-a...
4,008b9e19-c1d3-4c9c-af36-791ec4cc27e8,474eec2a-3b41-4e9b-b286-de7e12453b3e,\N,aliquot,e505cac9-a568-42db-bc67-d90af57e5265,General,Notification,2022-08-08T12:45:37.159434-05:00,Approved,Some RNA-Seq file types may be missing from th...,data/raw/tcga/star_counts/008b9e19-c1d3-4c9c-a...


In [13]:
# =============================================================================
# Validate annotation consistency and summarize annotation scope
# =============================================================================

annotation_metadata_columns = [
    "submitter_id",
    "entity_type",
    "entity_id",
    "category",
    "classification",
    "created_datetime",
    "status",
    "notes",
]

annotation_consistency_matrix = (
    file_annotations_df
    .groupby("annotation_id")[annotation_metadata_columns]
    .nunique(dropna=False)
)

inconsistent_annotation_ids = annotation_consistency_matrix.index[
    (annotation_consistency_matrix > 1).any(axis=1)
].tolist()

if inconsistent_annotation_ids:
    raise ValueError(
        "Repeated annotation IDs contain inconsistent metadata: "
        f"{len(inconsistent_annotation_ids):,} annotation IDs."
    )

unique_annotations_df = (
    file_annotations_df
    .drop_duplicates(subset="annotation_id")
    .copy()
)

annotation_file_counts = (
    file_annotations_df
    .groupby("annotation_id")["file_id"]
    .nunique()
)

annotations_linked_to_multiple_files = int(
    (annotation_file_counts > 1).sum()
)

entity_type_summary = (
    file_annotations_df
    .groupby("entity_type", as_index=False)
    .agg(
        file_annotation_associations=("annotation_id", "size"),
        unique_annotations=("annotation_id", "nunique"),
        affected_files=("file_id", "nunique"),
        affected_entities=("entity_id", "nunique"),
    )
    .sort_values(
        ["affected_files", "unique_annotations"],
        ascending=False,
    )
)

classification_summary = (
    file_annotations_df
    .groupby("classification", as_index=False)
    .agg(
        file_annotation_associations=("annotation_id", "size"),
        unique_annotations=("annotation_id", "nunique"),
        affected_files=("file_id", "nunique"),
        affected_entities=("entity_id", "nunique"),
    )
    .sort_values(
        ["affected_files", "unique_annotations"],
        ascending=False,
    )
)

category_summary = (
    file_annotations_df
    .groupby(
        ["classification", "category"],
        as_index=False,
    )
    .agg(
        file_annotation_associations=("annotation_id", "size"),
        unique_annotations=("annotation_id", "nunique"),
        affected_files=("file_id", "nunique"),
        affected_entities=("entity_id", "nunique"),
    )
    .sort_values(
        ["affected_files", "unique_annotations"],
        ascending=False,
    )
)

status_summary = (
    unique_annotations_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="unique_annotations")
)

print("Annotation consistency validated.")
print(
    "Annotations linked to multiple RNA-seq files:",
    f"{annotations_linked_to_multiple_files:,}",
)
print(
    "Distinct entity types:",
    f"{file_annotations_df['entity_type'].nunique():,}",
)
print(
    "Distinct classifications:",
    f"{file_annotations_df['classification'].nunique():,}",
)
print(
    "Distinct categories:",
    f"{file_annotations_df['category'].nunique():,}",
)

print("\nAnnotation status:")
display(status_summary)

print("\nEntity-type summary:")
display(entity_type_summary)

print("\nClassification summary:")
display(classification_summary)

print("\nComplete classification/category summary:")

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    100,
):
    display(category_summary)

Annotation consistency validated.
Annotations linked to multiple RNA-seq files: 26
Distinct entity types: 5
Distinct classifications: 3
Distinct categories: 25

Annotation status:


,status,unique_annotations
0,Approved,2614



Entity-type summary:


,entity_type,file_annotation_associations,unique_annotations,affected_files,affected_entities
2,case,1053,1025,870,847
0,aliquot,1560,1560,801,801
1,analyte,28,24,27,23
4,sample,4,4,4,4
3,portion,1,1,1,1



Classification summary:


,classification,file_annotation_associations,unique_annotations,affected_files,affected_entities
1,Notification,2315,2283,1420,1482
0,CenterNotification,283,283,155,155
2,Observation,48,48,43,47



Complete classification/category summary:


,classification,category,file_annotation_associations,unique_annotations,affected_files,affected_entities
8,Notification,General,1262,1262,631,631
18,Notification,Prior malignancy,394,385,387,378
4,Notification,Alternate sample pipeline,151,151,151,151
10,Notification,History of unacceptable prior treatment related to a prior/other malignancy,135,132,133,130
2,CenterNotification,Item flagged DNU,128,128,128,128
0,CenterNotification,Center QC failed,117,117,117,117
21,Notification,Synchronous malignancy,109,106,109,106
15,Notification,Neoadjuvant therapy,65,62,61,59
9,Notification,History of acceptable prior treatment related to a prior/other malignancy,46,46,46,46
1,CenterNotification,Item Flagged Low Quality,38,38,38,38


In [15]:
# =============================================================================
# Validate MD5 checksums for all downloaded RNA-seq payloads
# =============================================================================

if not payload_validation_df["payload_present"].all():
    raise RuntimeError(
        "MD5 validation cannot start while expected payloads are missing."
    )

if not payload_validation_df["size_matches"].all():
    raise RuntimeError(
        "MD5 validation cannot start while file-size mismatches remain."
    )


def calculate_md5(
    file_path: Path,
    chunk_size: int = 8 * 1024**2,
) -> str:
    """Calculate an MD5 checksum for file-integrity validation."""
    md5_hash = hashlib.md5(usedforsecurity=False)

    with file_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            md5_hash.update(chunk)

    return md5_hash.hexdigest()


checksum_validation_df = payload_validation_df[
    [
        "file_id",
        "expected_file_name",
        "expected_md5",
        "expected_size_bytes",
        "local_path",
        "local_size_bytes",
    ]
].copy()

observed_md5_values = []
n_payloads = len(checksum_validation_df)

print(f"Starting MD5 validation for {n_payloads:,} payloads...")

for position, file_path in enumerate(
    checksum_validation_df["local_path"],
    start=1,
):
    observed_md5_values.append(
        calculate_md5(Path(file_path))
    )

    if position % 500 == 0 or position == n_payloads:
        print(
            f"Hashed {position:,} / {n_payloads:,} payloads",
            flush=True,
        )

checksum_validation_df["observed_md5"] = observed_md5_values

checksum_validation_df["md5_matches"] = (
    checksum_validation_df["expected_md5"].str.lower()
    == checksum_validation_df["observed_md5"].str.lower()
)

checksum_mismatches_df = checksum_validation_df.loc[
    ~checksum_validation_df["md5_matches"]
].copy()

print("\nMD5 validation completed.")
print(f"Payloads checked:       {len(checksum_validation_df):,}")
print(
    f"Matching MD5 checksums: "
    f"{checksum_validation_df['md5_matches'].sum():,}"
)
print(f"MD5 mismatches:         {len(checksum_mismatches_df):,}")

if not checksum_mismatches_df.empty:
    display(
        checksum_mismatches_df[
            [
                "file_id",
                "expected_file_name",
                "expected_md5",
                "observed_md5",
            ]
        ]
    )

    raise ValueError(
        "Downloaded TCGA RNA-seq payloads failed MD5 validation."
    )

print("All downloaded payloads match the frozen GDC manifest.")

Starting MD5 validation for 10,308 payloads...
Hashed 500 / 10,308 payloads
Hashed 1,000 / 10,308 payloads
Hashed 1,500 / 10,308 payloads
Hashed 2,000 / 10,308 payloads
Hashed 2,500 / 10,308 payloads
Hashed 3,000 / 10,308 payloads
Hashed 3,500 / 10,308 payloads
Hashed 4,000 / 10,308 payloads
Hashed 4,500 / 10,308 payloads
Hashed 5,000 / 10,308 payloads
Hashed 5,500 / 10,308 payloads
Hashed 6,000 / 10,308 payloads
Hashed 6,500 / 10,308 payloads
Hashed 7,000 / 10,308 payloads
Hashed 7,500 / 10,308 payloads
Hashed 8,000 / 10,308 payloads
Hashed 8,500 / 10,308 payloads
Hashed 9,000 / 10,308 payloads
Hashed 9,500 / 10,308 payloads
Hashed 10,000 / 10,308 payloads
Hashed 10,308 / 10,308 payloads

MD5 validation completed.
Payloads checked:       10,308
Matching MD5 checksums: 10,308
MD5 mismatches:         0
All downloaded payloads match the frozen GDC manifest.


In [16]:
# =============================================================================
# Write consolidated GDC annotations for downstream cohort construction and QC
# =============================================================================

ANNOTATION_OUTPUT_DIR = Paths.interim / "metadata"

ANNOTATION_OUTPUT_PATH = (
    ANNOTATION_OUTPUT_DIR
    / "tcga_primary_tumor_rnaseq_star_counts_file_annotations.tsv"
)

ANNOTATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

file_annotations_output_df = (
    file_annotations_df
    .sort_values(
        ["file_id", "annotation_id"]
    )
    .reset_index(drop=True)
)

file_annotations_output_df.to_csv(
    ANNOTATION_OUTPUT_PATH,
    sep="\t",
    index=False,
)

annotation_output_size_mib = (
    ANNOTATION_OUTPUT_PATH.stat().st_size / 1024**2
)

annotation_output_relative_path = (
    ANNOTATION_OUTPUT_PATH.resolve()
    .relative_to(Paths.root.resolve())
    .as_posix()
)

print("Consolidated GDC annotations written.")
print(f"Output:                   {annotation_output_relative_path}")
print(f"Rows:                     {len(file_annotations_output_df):,}")
print(
    f"Unique annotation IDs:    "
    f"{file_annotations_output_df['annotation_id'].nunique():,}"
)
print(
    f"Annotated RNA-seq files:  "
    f"{file_annotations_output_df['file_id'].nunique():,}"
)
print(f"File size:                {annotation_output_size_mib:,.3f} MiB")

Consolidated GDC annotations written.
Output:                   data/interim/metadata/tcga_primary_tumor_rnaseq_star_counts_file_annotations.tsv
Rows:                     2,646
Unique annotation IDs:    2,614
Annotated RNA-seq files:  1,603
File size:                0.983 MiB


## Summary

The local TCGA primary-tumor RNA-seq STAR Counts download was validated against the version-controlled frozen GDC manifest.

### Download integrity

- 10,308 manifest-listed file IDs were expected.
- 10,308 corresponding UUID directories were present.
- All 10,308 expected RNA-seq payloads were present.
- No missing or unexpected UUID directories were detected.
- No missing payloads, file-size mismatches, or partial download files were detected.
- The expected and observed payload sizes were both 40.672680 GiB.
- All 10,308 downloaded payloads matched their manifest-recorded MD5 checksums.

These results establish that the local RNA-seq acquisition is complete and byte-level consistent with the frozen GDC manifest.

### Auxiliary GDC files

Each UUID directory contained a GDC Client `logs` directory used for transfer-state management. These directories are not biological inputs and will not be included in downstream expression processing.

A total of 1,603 UUID directories also contained an `annotations.txt` file:

- 2,646 file-to-annotation associations were identified;
- 2,614 unique annotation IDs were represented;
- 1,676 distinct annotated entities were represented;
- all annotations had `Approved` status;
- annotations covered five entity types, three classifications, and 25 distinct categories.

The annotation set includes technical quality warnings, protocol-related notices, clinical-context annotations, and general observations. No samples were excluded in this notebook because annotation interpretation requires category-, entity-, and context-specific rules.

The consolidated annotation table was written to:

- `data/interim/metadata/tcga_primary_tumor_rnaseq_star_counts_file_annotations.tsv`

This table is retained as a downstream consumable for TCGA cohort construction and RNA-seq quality control.

### Conclusion

The TCGA RNA-seq download is complete and integrity-validated. The raw payloads remain immutable. Expression-table parsing, sample selection, annotation-based eligibility decisions, and expression quality control will be handled explicitly in downstream TCGA notebooks.